# Olist End-to-End Business Data Analytics
## Notebook 01 — Data Wrangling & Cleaning

This notebook prepares the two Olist datasets used in the project:

1. **Brazilian E-Commerce Public Dataset by Olist**
2. **Marketing Funnel by Olist**

The goal is to build a reliable analytical foundation before starting the Exploratory Data Analysis (EDA).

### This notebook covers

- Understanding the structure of each dataset
- Validating relationships between tables
- Checking data quality
- Identifying missing values
- Identifying exact and key-level duplicates
- Correcting data types
- Standardising categorical and text fields
- Making explicit cleaning decisions
- Preparing clean relational tables for the next notebook
- Validating how the Marketing Funnel connects to the E-Commerce ecosystem through `seller_id`

In [1]:
# Import libraries

import numpy as np
import pandas as pd 

In [2]:
# Define data paths

RAW_PATH = "../data/raw/"
PROCESSED_PATH = "../data/processed/"

## 2. Load the raw datasets

The project uses 11 raw tables: nine from the Olist E-Commerce dataset and two from the Marketing Funnel dataset. Each table is loaded separately to keep the relational structure explicit and easy to follow throughout the analysis.

### E-Commerce
- Customers
- Orders
- Order items
- Payments
- Reviews
- Products
- Sellers
- Geolocation
- Product category translation

### Marketing Funnel
- Marketing Qualified Leads (MQLs)
- Closed deals

In [3]:
# Load the 1st dataset and see if it's working to load the rest 
customers = pd.read_csv(RAW_PATH + "olist_customers_dataset.csv")

customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [4]:
# Load E-Commerce datasets

orders = pd.read_csv(RAW_PATH + "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW_PATH + "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW_PATH + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW_PATH + "olist_order_reviews_dataset.csv")
products = pd.read_csv(RAW_PATH + "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_PATH + "olist_sellers_dataset.csv")
geolocation = pd.read_csv(RAW_PATH + "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(RAW_PATH + "product_category_name_translation.csv")

# Load Marketing Funnel datasets

mql = pd.read_csv( RAW_PATH + "olist_marketing_qualified_leads_dataset.csv")

closed_deals = pd.read_csv(RAW_PATH + "olist_closed_deals_dataset.csv")

## 3. Initial data understanding

Before changing anything, we inspect the size, columns and data types of each table. This helps us understand the granularity of the datasets and identify potential data-quality issues before cleaning.


- How many observations and variables does each dataset contain?
- What does each row represent?
- Which columns are available?
- Are the current data types appropriate?
- Where are missing values present?

### 3.1 Dataset dimensions

In [5]:
# Check the shape of each dataset

print("E-COMMERCE DATASETS")
print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Order items:", order_items.shape)
print("Payments:", payments.shape)
print("Reviews:", reviews.shape)
print("Products:", products.shape)
print("Sellers:", sellers.shape)
print("Geolocation:", geolocation.shape)
print("Category translation:", category_translation.shape)

print("\nMARKETING FUNNEL DATASETS")
print("Marketing Qualified Leads:", mql.shape)
print("Closed deals:", closed_deals.shape)

E-COMMERCE DATASETS
Customers: (99441, 5)
Orders: (99441, 8)
Order items: (112650, 7)
Payments: (103886, 5)
Reviews: (99224, 7)
Products: (32951, 9)
Sellers: (3095, 4)
Geolocation: (1000163, 5)
Category translation: (71, 2)

MARKETING FUNNEL DATASETS
Marketing Qualified Leads: (8000, 4)
Closed deals: (842, 14)


### 3.2 Preview the data

In [6]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [7]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [8]:
mql.head()

,mql_id,first_contact_date,landing_page_id,origin
0,dac32acd4db4c29c230538b72f8dd87d,2018-02-01,88740e65d5d6b056e0cda098e1ea6313,social
1,8c18d1de7f67e60dbd64e3c07d7e9d5d,2017-10-20,007f9098284a86ee80ddeb25d53e0af8,paid_search
2,b4bc852d233dfefc5131f593b538befa,2018-03-22,a7982125ff7aa3b2054c6e44f9d28522,organic_search
3,6be030b81c75970747525b843c1ef4f8,2018-01-22,d45d558f0daeecf3cccdffe3c59684aa,email
4,5420aad7fec3549a85876ba1c529bd84,2018-02-21,b48ec5f3b04e9068441002a19df93c6c,organic_search


In [9]:
closed_deals.head()

,mql_id,seller_id,sdr_id,sr_id,won_date,business_segment,lead_type,lead_behaviour_profile,has_company,has_gtin,average_stock,business_type,declared_product_catalog_size,declared_monthly_revenue
0,5420aad7fec3549a85876ba1c529bd84,2c43fb513632d29b3b58df74816f1b06,a8387c01a09e99ce014107505b92388c,4ef15afb4b2723d8f3d81e51ec7afefe,2018-02-26 19:58:54,pet,online_medium,cat,NaN,NaN,NaN,reseller,NaN,0.0
1,a555fb36b9368110ede0f043dfc3b9a0,bbb7d7893a450660432ea6652310ebb7,09285259593c61296eef10c734121d5b,d3d1e91a157ea7f90548eef82f1955e3,2018-05-08 20:17:59,car_accessories,industry,eagle,NaN,NaN,NaN,reseller,NaN,0.0
2,327174d3648a2d047e8940d7d15204ca,612170e34b97004b3ba37eae81836b4c,b90f87164b5f8c2cfa5c8572834dbe3f,6565aa9ce3178a5caf6171827af3a9ba,2018-06-05 17:27:23,home_appliances,online_big,cat,NaN,NaN,NaN,reseller,NaN,0.0
3,f5fee8f7da74f4887f5bcae2bafb6dd6,21e1781e36faf92725dde4730a88ca0f,56bf83c4bb35763a51c2baab501b4c67,d3d1e91a157ea7f90548eef82f1955e3,2018-01-17 13:51:03,food_drink,online_small,NaN,NaN,NaN,NaN,reseller,NaN,0.0
4,ffe640179b554e295c167a2f6be528e0,ed8cb7b190ceb6067227478e48cf8dde,4b339f9567d060bcea4f5136b9f5949e,d3d1e91a157ea7f90548eef82f1955e3,2018-07-03 20:17:45,home_appliances,industry,wolf,NaN,NaN,NaN,manufacturer,NaN,0.0


In [10]:
# Group datasets for quick inspection

all_datasets = [customers, orders, order_items, payments, reviews, products, sellers, geolocation, category_translation, mql, closed_deals]

dataset_names = ["Customers","Orders","Order items","Payments","Reviews","Products","Sellers","Geolocation","Category translation","Marketing Qualified Leads","Closed deals"]

### 3.3 Data types and missing values

In [11]:
# Inspect columns, data types and missing values

for name, df in zip(dataset_names, all_datasets):
    print("\n", name)
    df.info()


 Customers
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 11.0 MB

 Orders
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved

## 4. Understand the relational data model

Olist is not one flat table. It is a relational dataset, and different tables have different levels of granularity.

### Core E-Commerce relationships

- `customers.customer_id` → `orders.customer_id`
- `orders.order_id` → `order_items.order_id`
- `orders.order_id` → `payments.order_id`
- `orders.order_id` → `reviews.order_id`
- `order_items.product_id` → `products.product_id`
- `order_items.seller_id` → `sellers.seller_id`
- `products.product_category_name` → `category_translation.product_category_name`

### Marketing Funnel relationships

- `marketing_qualified_leads.mql_id` → `closed_deals.mql_id`
- `closed_deals.seller_id` → `sellers.seller_id`
- `closed_deals.seller_id` → `order_items.seller_id`

This last connection is strategically important because it allows us to follow a seller from **lead acquisition → closed deal → marketplace activity**.

> **Important:** we should not merge every table into one giant DataFrame. One order can contain multiple items, payments and potentially multiple seller records. A careless merge could multiply rows and distort KPIs.


### 4.1 Check primary keys

In [12]:
# Check primary keys

print("Customers - customer_id unique:", customers["customer_id"].is_unique)
print("Orders - order_id unique:", orders["order_id"].is_unique)
print("Products - product_id unique:", products["product_id"].is_unique)
print("Sellers - seller_id unique:", sellers["seller_id"].is_unique)
print("MQL - mql_id unique:", mql["mql_id"].is_unique)
print("Closed deals - mql_id unique:", closed_deals["mql_id"].is_unique)
print("Closed deals - seller_id unique:", closed_deals["seller_id"].is_unique)

Customers - customer_id unique: True
Orders - order_id unique: True
Products - product_id unique: True
Sellers - seller_id unique: True
MQL - mql_id unique: True
Closed deals - mql_id unique: True
Closed deals - seller_id unique: True


### 4.2 Check relationships between tables

In [13]:
# Check key relationships

print("Orders linked to customers:", orders["customer_id"].isin(customers["customer_id"]).mean())

print("Order items linked to orders:", order_items["order_id"].isin(orders["order_id"]).mean())

print("Order items linked to products:", order_items["product_id"].isin(products["product_id"]).mean())

print("Order items linked to sellers:", order_items["seller_id"].isin(sellers["seller_id"]).mean())

print("Closed deals linked to MQLs:", closed_deals["mql_id"].isin(mql["mql_id"]).mean())

print("Closed-deal sellers linked to E-Commerce sellers:", closed_deals["seller_id"].isin(sellers["seller_id"]).mean())

Orders linked to customers: 1.0
Order items linked to orders: 1.0
Order items linked to products: 1.0
Order items linked to sellers: 1.0
Closed deals linked to MQLs: 1.0
Closed-deal sellers linked to E-Commerce sellers: 0.4513064133016627


### Findings

The main E-Commerce relationships show full key coverage, confirming that orders, customers, products and sellers can be reliably connected through their corresponding IDs.

The Marketing Funnel also shows a complete relationship between MQLs and closed deals. However, only **45.13% of sellers in the closed-deals dataset are found in the E-Commerce sellers table**.

This is not treated as a data-quality error. Instead, it defines the scope of the later analysis: seller performance can only be evaluated for the subset of Marketing Funnel sellers that can be matched to the E-Commerce data.

With the main relationships understood, the next step is to assess the quality of the data before applying any transformations.

## 5. Data Quality & Cleaning

Now that the structure and relationships between the datasets are understood, we can assess the quality of the data and prepare it for analysis.

The cleaning process will focus on:

- Correcting inconsistent or misspelled column names.
- Identifying and understanding missing values.
- Checking for duplicate records.
- Correcting data types where necessary.
- Checking categorical values for inconsistencies.

Cleaning decisions will be made according to the meaning of each variable rather than applying transformations automatically.

### 5.1 Column names 

The datasets already follow a consistent snake_case naming convention. However, two columns in the `products` dataset contain a spelling error (`lenght` instead of `length`).

In [14]:
# Correct spelling errors in product column names

products.rename(columns={ "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"}, inplace=True)

products.columns

Index(['product_id', 'product_category_name', 'product_name_length',
       'product_description_length', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='str')

### 5.2 Missing values

During the initial inspection with `.info()`, we observed that some datasets contain columns with fewer non-null values than total rows, particularly `orders`, `reviews`, `products`, `mql`, and `closed_deals`.

Before deciding how to handle them, we first need to quantify the missing values in each dataset and understand where they are concentrated.

In [15]:
# Check missing values in all datasets

for name, df in zip(dataset_names, all_datasets):
    print("\n", name)
    print(df.isnull().sum())


 Customers
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

 Orders
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

 Order items
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

 Payments
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

 Reviews
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
r

#### Initial observations

Missing values are not present across all datasets. Customers, order items, payments, sellers, geolocation and category translation contain no missing values.

Missing data is concentrated in five datasets:

- **Orders:** missing values appear in approval and delivery dates.
- **Reviews:** missing values are concentrated in the two review comment fields.
- **Products:** 610 records are missing category and descriptive information, while only 2 records are missing physical product dimensions.
- **Marketing Qualified Leads:** 60 leads have no recorded acquisition origin.
- **Closed Deals:** several seller attributes contain missing values, with particularly high missingness in `has_company`, `has_gtin`, `average_stock`, and `declared_product_catalog_size`.

These missing values should be investigated separately because they may have different meanings and may require different cleaning decisions.

### 5.2.1 Investigate missing values in Orders

The initial missing-value check identified missing values in three order lifecycle fields: `order_approved_at`, `order_delivered_carrier_date`, and `order_delivered_customer_date`.

Since these columns represent different stages of the order process, we first investigate whether their missing values are related to `order_status` before deciding if any cleaning is required.

In [16]:
# Check order status distribution

orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [17]:
# Check order status when approval date is missing

orders[orders["order_approved_at"].isnull()]["order_status"].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

In [18]:
# Check order status when carrier delivery date is missing

orders[orders["order_delivered_carrier_date"].isnull()]["order_status"].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

In [19]:
# Check order status when customer delivery date is missing

orders[orders["order_delivered_customer_date"].isnull()]["order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [20]:
# Inspect orders with missing customer delivery date

orders[orders["order_delivered_customer_date"].isnull()].head(10)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaN,NaN,2017-05-09 00:00:00
44,ee64d42b8cf066f35eac1cf57de1aa85,caded193e8e47b8362864762a83db3c5,shipped,2018-06-04 16:44:48,2018-06-05 04:31:18,2018-06-05 14:32:00,NaN,2018-06-28 00:00:00
103,0760a852e4e9d89eb77bf631eaaf1c84,d2a79636084590b7465af8ab374a8cf5,invoiced,2018-08-03 17:44:42,2018-08-07 06:15:14,NaN,NaN,2018-08-21 00:00:00
128,15bed8e2fec7fdbadb186b57c46c92f2,f3f0e613e0bdb9c7cee75504f0f90679,processing,2017-09-03 14:22:03,2017-09-03 14:30:09,NaN,NaN,2017-10-03 00:00:00
154,6942b8da583c2f9957e990d028607019,52006a9383bf149a4fb24226b173106f,shipped,2018-01-10 11:33:07,2018-01-11 02:32:30,2018-01-11 19:39:23,NaN,2018-02-07 00:00:00
162,36530871a5e80138db53bcfd8a104d90,4dafe3c841d2d6cc8a8b6d25b35704b9,shipped,2017-05-09 11:48:37,2017-05-11 11:45:14,2017-05-11 13:21:47,NaN,2017-06-08 00:00:00
231,4d630f57194f5aba1a3d12ce23e71cd9,6d491c9fe2f04f6e2af6ec033cd8907c,shipped,2017-11-17 19:53:21,2017-11-18 19:50:31,2017-11-22 17:28:34,NaN,2017-12-13 00:00:00
266,8e24261a7e58791d10cb1bf9da94df5c,64a254d30eed42cd0e6c36dddb88adf0,unavailable,2017-11-16 15:09:28,2017-11-16 15:26:57,NaN,NaN,2017-12-05 00:00:00
299,3b4ad687e7e5190db827e1ae5a8989dd,1a87b8517b7d31373b50396eb15cb445,shipped,2018-06-28 12:52:15,2018-06-28 13:11:09,2018-07-04 15:20:00,NaN,2018-08-03 00:00:00
305,b68d69564a79dea4776afa33d1d2fcab,de1e5517fb50896bbdcff5814fb31802,shipped,2018-02-28 08:57:03,2018-02-28 10:40:35,2018-03-05 16:10:13,NaN,2018-03-23 00:00:00


In [21]:
# Inspect delivered orders with missing customer delivery date

orders[(orders["order_status"] == "delivered") & (orders["order_delivered_customer_date"].isnull())]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


In [22]:
# Inspect delivered orders with missing approval date

orders[(orders["order_status"] == "delivered") &(orders["order_approved_at"].isnull())]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaN,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01 00:00:00
38290,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2017-02-19 01:28:47,NaN,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27 00:00:00
39334,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2017-02-18 11:04:19,NaN,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22 00:00:00
48401,7002a78c79c519ac54022d4f8a65e6e8,d5de688c321096d15508faae67a27051,delivered,2017-01-19 22:26:59,NaN,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16 00:00:00
61743,2eecb0d85f281280f79fa00f9cec1a95,a3d3c38e58b9d2dfb9207cab690b6310,delivered,2017-02-17 17:21:55,NaN,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20 00:00:00


In [23]:
# Inspect delivered orders with missing carrier delivery date

orders[(orders["order_status"] == "delivered") & (orders["order_delivered_carrier_date"].isnull())]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
73222,2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaN,2017-11-20 19:44:47,2017-11-14 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00


#### Findings and cleaning decision

Most missing order dates can be explained by the order status:

- Missing `order_delivered_customer_date` values are mostly found in orders that were not fully completed, such as `shipped`, `canceled`, `unavailable`, `invoiced`, or `processing`.
- Missing `order_delivered_carrier_date` values are also mostly related to orders that did not complete the shipping process.
- Most missing `order_approved_at` values belong to canceled orders.

We also found a small number of exceptions among orders marked as `delivered`:

- 14 have no approval date.
- 2 have no carrier delivery date.
- 8 have no customer delivery date.

After checking these cases, we can see that some orders still contain information from later stages of the process. This suggests that the missing dates are data-quality issues, but the orders themselves are still valid records.

**Cleaning decision:** We will keep these missing values and will not fill them with estimated dates. If a future analysis requires one of these dates, records without that specific date will be excluded only from that calculation.

### 5.2.2 Investigate missing values in Reviews

The initial missing-value check identified missing values in two review text fields: `review_comment_title` and `review_comment_message`.

Since every review contains a `review_score`, we investigate whether the missing text fields represent incomplete records or reviews where customers submitted a score without written feedback.

In [24]:
# Check completeness of review scores

print("Total reviews:", len(reviews))
print("Missing review scores:", reviews["review_score"].isnull().sum())

Total reviews: 99224
Missing review scores: 0


In [25]:
# Check reviews with missing written feedback

print("Missing title:",reviews["review_comment_title"].isnull().sum())

print("Missing message:",reviews["review_comment_message"].isnull().sum())

print("Missing both title and message:", (reviews["review_comment_title"].isnull() & reviews["review_comment_message"].isnull()).sum())

Missing title: 87656
Missing message: 58247
Missing both title and message: 56518


In [26]:
# Inspect reviews without written feedback

reviews[reviews["review_comment_title"].isnull() &reviews["review_comment_message"].isnull()].head(10)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
5,15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,NaN,NaN,2018-04-13 00:00:00,2018-04-16 00:39:37
6,07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,NaN,NaN,2017-07-16 00:00:00,2017-07-18 19:30:34
7,7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,NaN,NaN,2018-08-14 00:00:00,2018-08-14 21:36:06
8,a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,NaN,NaN,2017-05-17 00:00:00,2017-05-18 12:05:37
10,c9cfd2d5ab5911836ababae136c3a10c,cdf9aa68e72324eeb25c7de974696ee2,5,NaN,NaN,2017-12-23 00:00:00,2017-12-26 14:36:03
11,96052551d87e5f62e6c9f6974ec392e9,3d374c9e46530bb5ed4a7648915306a6,5,NaN,NaN,2017-12-19 00:00:00,2017-12-20 10:25:22
13,23f75a37effc35d9a915b4e1ad483793,2eaf8e099d871cd5c22b83b5ea8f6e0e,4,NaN,NaN,2018-03-28 00:00:00,2018-03-30 15:10:55


#### Findings and cleaning decision

All 99,224 reviews contain a `review_score`, while written feedback is frequently missing.

This shows that a review can contain a valid score without a title or written comment. The missing text therefore does not make the review itself incomplete.

For this project, customer satisfaction will be analyzed using `review_score`. The text fields `review_comment_title` and `review_comment_message` are not required because text or sentiment analysis is outside the scope of the project.

**Cleaning decision:** Keep all review records and `review_score`, but remove the two written-feedback columns from the cleaned dataset.

In [27]:
reviews = reviews.drop(columns=["review_comment_title", "review_comment_message"])

### 5.2.3 Investigate missing values in Products

The initial missing-value check identified two clear patterns in the `products` dataset.

Four product-description fields contain 610 missing values each, while the four product-dimension fields contain only 2 missing values each.

Before deciding how to clean them, we check whether these missing values belong to the same products and whether the affected fields are relevant to the business analysis.

In [28]:
# Inspect products with missing category

products[products["product_category_name"].isnull()].head(10)

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
244,e10758160da97891c2fdcbc35f0f031d,NaN,NaN,NaN,NaN,2200.0,16.0,2.0,11.0
294,39e3b9b12cd0bf8ee681bbc1c130feb5,NaN,NaN,NaN,NaN,300.0,16.0,7.0,11.0
299,794de06c32a626a5692ff50e4985d36f,NaN,NaN,NaN,NaN,300.0,18.0,8.0,14.0
347,7af3e2da474486a3519b0cba9dea8ad9,NaN,NaN,NaN,NaN,200.0,22.0,14.0,14.0
428,629beb8e7317703dcc5f35b5463fd20e,NaN,NaN,NaN,NaN,1400.0,25.0,25.0,25.0


In [29]:
# Count products missing all four description fields

products[products[["product_category_name", "product_name_length", "product_description_length", "product_photos_qty"]].isnull().all(axis=1)].shape[0]

610

In [30]:
# Inspect products with missing dimensions

products[products["product_weight_g"].isnull()]

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
# Count products missing all dimension fields

products[products[["product_weight_g", "product_length_cm", "product_height_cm","product_width_cm"]].isnull().all(axis=1)].shape[0]

2

In [32]:
# Check how many order items belong to products with missing category

missing_category_products = products[products["product_category_name"].isnull()]["product_id"]

missing_category_order_items = order_items[order_items["product_id"].isin(missing_category_products)]

print("Products with missing category:", len(missing_category_products))
print("Order items from these products:", len(missing_category_order_items))

Products with missing category: 610
Order items from these products: 1603


In [33]:
# Count unique products with missing category that appear in order items

unique_missing_products_sold = missing_category_order_items["product_id"].nunique()

print("Products with missing category:", len(missing_category_products))
print("Unique products sold:", unique_missing_products_sold)

Products with missing category: 610
Unique products sold: 610


In [34]:
#Check sales value of products with missing categories to understand what part of the sales value we would leave unclassified if ignore these products

print("Sales value from products with missing category:",missing_category_order_items["price"].sum())

Sales value from products with missing category: 179535.28


In [35]:
total_sales = order_items["price"].sum()
missing_category_sales = missing_category_order_items["price"].sum()

print("Total sales value:", total_sales)
print("Missing-category sales value:", missing_category_sales)
print("Share of sales:",(missing_category_sales / total_sales) * 100)

Total sales value: 13591643.7
Missing-category sales value: 179535.28
Share of sales: 1.3209239733086882


#### Findings and cleaning decision

The 610 products with missing `product_category_name` also have missing values in `product_name_length`, `product_description_length`, and `product_photos_qty`.

All 610 products appear in `order_items`, representing 1,603 order items and approximately 1.32% of the total sales value. Therefore, these are valid products with real marketplace activity, despite having incomplete product information.

Only 2 products have missing physical dimensions, which represents a very small number of records.

**Cleaning decision:**

- Keep the 610 products and their related transactions.
- Replace missing `product_category_name` values with `"unknown"` so these sales can remain visible in category-level analysis.
- Keep the missing descriptive fields as `NaN`, since their values cannot be reliably inferred and they are not required for the main business analysis.
- Keep the 2 products with missing dimensions and leave those values as `NaN`.

In [36]:
# Replace missing product categories with "unknown"

products["product_category_name"] = products["product_category_name"].fillna("unknown")

### 5.2.4 Investigate missing values in Marketing Qualified Leads

The initial missing-value check identified 60 missing values in `origin`.

The `origin` field represents the acquisition channel of each Marketing Qualified Lead. Since this variable will be important for analyzing seller acquisition performance, we first investigate the missing records before deciding how to treat them.

In [37]:
# Check lead acquisition channels

mql["origin"].value_counts(dropna=False)

origin
organic_search       2296
paid_search          1586
social               1350
unknown              1099
direct_traffic        499
email                 493
referral              284
other                 150
display               118
other_publicities      65
NaN                    60
Name: count, dtype: int64

In [38]:
# Identify leads with missing acquisition origin

missing_origin_mql = mql[mql["origin"].isnull()]

print("MQLs with missing origin:", len(missing_origin_mql))

MQLs with missing origin: 60


In [39]:
# Check how many leads with missing origin became closed deals

missing_origin_closed = missing_origin_mql[missing_origin_mql["mql_id"].isin(closed_deals["mql_id"])]

print("Missing-origin MQLs:", len(missing_origin_mql))
print("Closed deals from missing-origin MQLs:", len(missing_origin_closed))

Missing-origin MQLs: 60
Closed deals from missing-origin MQLs: 14


In [40]:
# Inspect closed deals from MQLs with missing origin

closed_deals[closed_deals["mql_id"].isin(missing_origin_closed["mql_id"])].head(14)

,mql_id,seller_id,sdr_id,sr_id,won_date,business_segment,lead_type,lead_behaviour_profile,has_company,has_gtin,average_stock,business_type,declared_product_catalog_size,declared_monthly_revenue
106,67c34b9630469b3c13b7982316ffe7a1,6e985a12b7b98f0a15b7bf90cc882f61,56bf83c4bb35763a51c2baab501b4c67,c638112b43f1d1b86dcabb0da720c901,2018-01-22 13:18:32,home_office_furniture,industry,NaN,NaN,NaN,NaN,reseller,NaN,0.0
318,5290b66ff9e0c1115614365d8e20f10c,ab2a1539d7cdbddcfb6b568447ed767d,9749123c950bf8363ace42cb1c2d0815,495d4e95a8cf8bbf8b432b612a2aa328,2018-08-17 14:41:00,party,online_medium,cat,True,True,20-50,reseller,500.0,20000.0
398,d2ac71782272659e7171150d20d59158,3387acafd8bea46d73fc50cc9f7e2a9a,34d40cdaf94010a1d05b0d6212f9e909,2695de1affa7750089c0455f8ce27021,2018-05-15 21:17:04,computers,online_big,cat,NaN,NaN,NaN,reseller,NaN,0.0
616,f5baaf0afe419681731ec3d30dafd954,fb6e76ca64c0cc2c2dee53219e685f4b,4b339f9567d060bcea4f5136b9f5949e,d3d1e91a157ea7f90548eef82f1955e3,2018-05-10 19:26:38,games_consoles,online_big,cat,NaN,NaN,NaN,reseller,NaN,0.0
634,2b35567523c67c1a9c32ef101ae7bc73,7dd3be74c6cd6800d809935ff47bd3bf,9e4d1098a3b0f5da39b0bc48f9876645,fbf4aef3f6915dc0c3c97d6812522f6a,2018-03-21 13:03:40,health_beauty,offline,cat,NaN,NaN,NaN,manufacturer,NaN,0.0
658,d62e62abe24dcaa94f9e1b3678477b51,c611f4ce9ce875bcc063fa97fd4d7d12,4b339f9567d060bcea4f5136b9f5949e,85fc447d336637ba1df43e793199fbc8,2018-03-19 20:39:25,construction_tools_house_garden,online_medium,cat,NaN,NaN,NaN,manufacturer,NaN,0.0
685,61681a30d47dc8b1ec5180ccdec26ba5,4a4da369ad50f14d48337aa52eb826f5,068066e24f0c643eb1d089c7dd20cd73,6565aa9ce3178a5caf6171827af3a9ba,2018-01-29 15:09:03,health_beauty,online_medium,NaN,NaN,NaN,NaN,reseller,NaN,0.0
699,33ce1e734d9d50629fa2c36769285d53,53be10ff134691e94a4089b41c75874f,2b63542749aa9caf15f21816da1db341,d3d1e91a157ea7f90548eef82f1955e3,2018-09-11 13:14:37,construction_tools_house_garden,online_small,wolf,True,True,50-200,reseller,400.0,130000.0
734,a6bf24ce0939b46b6536e02a3d244cc3,ba87e2cbb33c1412290b8ccc37301672,4b339f9567d060bcea4f5136b9f5949e,060c0a26f19f4d66b42e0d8796688490,2018-03-07 14:58:07,food_drink,industry,wolf,NaN,NaN,NaN,manufacturer,NaN,0.0
743,baf5ecd84c6a8766519b98f66eec1511,e5d7bbbca541b3b39e1649f94074b963,b90f87164b5f8c2cfa5c8572834dbe3f,4ef15afb4b2723d8f3d81e51ec7afefe,2018-06-11 21:01:55,health_beauty,online_big,cat,NaN,NaN,NaN,reseller,NaN,0.0


In [41]:
# Calculate conversion rate for MQLs with missing origin

missing_origin_conversion_rate = (len(missing_origin_closed) / len(missing_origin_mql)) * 100

print("Missing-origin conversion rate:", missing_origin_conversion_rate)

Missing-origin conversion rate: 23.333333333333332


In [42]:
# Label missing acquisition origin

mql["origin"] = mql["origin"].fillna("missing")

#### Findings and cleaning decision

There are 60 MQLs with missing `origin`. Of these, 14 appear in `closed_deals`, meaning that 23.33% of the leads with missing acquisition information converted into closed deals.

Inspection of these 14 records shows valid seller and deal information, so the missing `origin` does not make these records invalid.

The dataset already contains an `"unknown"` origin category. Since we cannot confirm that `"unknown"` and missing values represent the same situation, they will remain separate.

**Cleaning decision:** Keep all 60 MQLs and replace missing `origin` values with `"missing"`. This preserves valid leads and conversions without assuming their acquisition channel.

### 5.2.5 Investigate missing values in Closed Deals

The `closed_deals` dataset contains 842 converted MQLs and includes information about the sellers acquired through the Marketing Funnel.

The initial missing-value check identified different levels of missingness across several seller and lead attributes. Before cleaning these fields, we investigate how much information is available and whether each variable is relevant to the business analysis.

In [43]:
# Check missing values and percentage in closed deals

closed_deals_missing = pd.DataFrame({"missing_values": closed_deals.isnull().sum(),"missing_percentage": closed_deals.isnull().mean() * 100})

closed_deals_missing

,missing_values,missing_percentage
mql_id,0,0.000000
seller_id,0,0.000000
sdr_id,0,0.000000
sr_id,0,0.000000
won_date,0,0.000000
business_segment,1,0.118765
lead_type,6,0.712589
lead_behaviour_profile,177,21.021378
has_company,779,92.517815
has_gtin,778,92.399050


#### Missing-value patterns

The missing-value summary shows different levels of missingness across the `closed_deals` dataset.

The variables can be grouped into three main patterns:

- **Low missingness:** `business_segment`, `lead_type`, and `business_type` have less than 2% missing values.
- **Moderate missingness:** `lead_behaviour_profile` is missing in approximately 21% of the records.
- **High missingness:** `has_company`, `has_gtin`, `average_stock`, and `declared_product_catalog_size` are missing in more than 90% of the records.

Because these groups have very different levels of data availability, they should not automatically receive the same cleaning treatment.

We will investigate the highly incomplete variables first to determine whether the available information is useful enough to keep them in the cleaned dataset.

In [44]:
# Inspect values available in highly incomplete columns

high_missing_cols = ["has_company", "has_gtin", "average_stock", "declared_product_catalog_size"]

for col in high_missing_cols:
    print(closed_deals[col].value_counts(dropna=False).head(10))

has_company
NaN      779
True      58
False      5
Name: count, dtype: int64
has_gtin
NaN      778
True      54
False     10
Name: count, dtype: int64
average_stock
NaN        776
5-20        22
50-200      15
1-5         10
20-50        8
200+         7
unknown      4
Name: count, dtype: int64
declared_product_catalog_size
NaN       773
100.0       9
50.0        7
300.0       5
400.0       4
20.0        4
1000.0      3
10.0        3
15.0        2
120.0       2
Name: count, dtype: int64


In [45]:
# Check whether highly incomplete seller attributes are missing together

closed_deals[high_missing_cols].notnull().sum(axis=1).value_counts().sort_index()

0    756
1     12
2     10
3     26
4     38
Name: count, dtype: int64

Most of these seller attributes have very limited information:

- 756 of 842 sellers have no data in any of the four fields.
- Only 38 sellers have all four fields completed.
- `average_stock` and `declared_product_catalog_size` could be useful for understanding seller potential, but there is not enough data to use them reliably in the main analysis.

**Decision:** Keep the columns for now and decide later whether they are useful for the final analysis.

In [46]:
# Inspect lead behaviour profile

closed_deals["lead_behaviour_profile"].value_counts(dropna=False)

lead_behaviour_profile
cat            407
NaN            177
eagle          123
wolf            95
shark           24
cat, wolf        8
eagle, wolf      3
eagle, cat       3
shark, cat       1
shark, wolf      1
Name: count, dtype: int64

In [47]:
# Check lead types when behaviour profile is missing

closed_deals[closed_deals["lead_behaviour_profile"].isnull()]["lead_type"].value_counts(dropna=False)

lead_type
online_medium      67
industry           29
online_small       20
online_big         20
offline            18
online_beginner    15
online_top          4
NaN                 4
Name: count, dtype: int64

In [48]:
# Check business segments when behaviour profile is missing

closed_deals[closed_deals["lead_behaviour_profile"].isnull()]["business_segment"].value_counts(dropna=False).head(10)

business_segment
home_decor                         25
health_beauty                      23
household_utilities                14
construction_tools_house_garden    12
car_accessories                    11
food_drink                          9
audio_video_electronics             9
food_supplement                     8
computers                           8
bed_bath_table                      6
Name: count, dtype: int64

In [49]:
# Label missing categorical seller attributes

closed_deals["lead_behaviour_profile"] = (closed_deals["lead_behaviour_profile"].fillna("missing"))

In [50]:
# Inspect values available in Low incomplete columns

low_missing_cols = ["business_segment","lead_type","business_type"]

for col in low_missing_cols:
    print(closed_deals[col].value_counts(dropna=False))

business_segment
home_decor                         105
health_beauty                       93
car_accessories                     77
household_utilities                 71
construction_tools_house_garden     69
audio_video_electronics             64
computers                           34
pet                                 30
food_supplement                     28
food_drink                          26
sports_leisure                      25
bed_bath_table                      22
bags_backpacks                      22
toys                                20
fashion_accessories                 19
home_office_furniture               14
stationery                          13
phone_mobile                        13
small_appliances                    12
handcrafted                         12
baby                                10
music_instruments                    9
books                                9
watches                              8
jewerly                              8
home_app

In [51]:
# Label missing categorical seller attributes

cols_to_fill = ["business_segment","lead_type","business_type"]

for col in cols_to_fill:
    closed_deals[col] = closed_deals[col].fillna("missing")

#### Findings

`lead_behaviour_profile` is missing for 177 sellers. These missing values are distributed across different lead types and business segments, so there is no clear basis for assigning a specific profile.

The remaining categorical variables have very few missing values: 1 in `business_segment`, 6 in `lead_type`, and 10 in `business_type`.

**Decision:** Keep all seller records and replace these missing categorical values with `"missing"`. No categories will be inferred.

## 6. Duplicate analysis

We distinguish between:

1. **Exact duplicate rows** — the whole row is repeated.
2. **Repeated IDs caused by valid one-to-many relationships**.

For example, repeated `order_id` values in `order_items` are expected because one order can contain several products. Removing them would destroy real transaction information.


In [52]:
# Check exact duplicate rows in all datasets

for name, df in zip(dataset_names, all_datasets):
    duplicate_count = df.duplicated().sum()
    print(f"{name}: {duplicate_count} duplicate rows")

Customers: 0 duplicate rows
Orders: 0 duplicate rows
Order items: 0 duplicate rows
Payments: 0 duplicate rows
Reviews: 0 duplicate rows
Products: 0 duplicate rows
Sellers: 0 duplicate rows
Geolocation: 261831 duplicate rows
Category translation: 0 duplicate rows
Marketing Qualified Leads: 0 duplicate rows
Closed deals: 0 duplicate rows


In [53]:
# Inspect exact duplicate rows in geolocation

geolocation[geolocation.duplicated(keep=False)].sort_values(["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"]).head(20)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
519,1001,-23.551337,-46.634027,sao paulo,SP
583,1001,-23.551337,-46.634027,sao paulo,SP
818,1001,-23.551337,-46.634027,sao paulo,SP
206,1001,-23.550498,-46.634338,sao paulo,SP
429,1001,-23.550498,-46.634338,sao paulo,SP
596,1001,-23.550498,-46.634338,sao paulo,SP
639,1001,-23.550498,-46.634338,sao paulo,SP
771,1001,-23.550498,-46.634338,sao paulo,SP
912,1001,-23.550498,-46.634338,sao paulo,SP
985,1001,-23.550498,-46.634338,sao paulo,SP


No exact duplicate rows were found in 10 of the 11 datasets.

The `geolocation` dataset contains **261,831 exact duplicate rows**. Inspection confirms that these records repeat the same zip code, coordinates, city and state. Since these duplicated rows do not add new geographic information, they can be removed from the cleaned dataset.

In [54]:
# Check how many different coordinates can exist within the same zip code

geo_coordinates = (geolocation[["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"]].drop_duplicates())

geo_coordinates.groupby("geolocation_zip_code_prefix").size().sort_values(ascending=False).head(10)

geolocation_zip_code_prefix
38400    746
11680    727
35500    726
11740    666
36400    627
39400    620
35162    611
38408    600
37200    595
35900    589
dtype: int64

#### Findings and cleaning decision

Exact duplicate rows were only found in the `geolocation` dataset. Further inspection shows that the same `geolocation_zip_code_prefix` can have many different latitude and longitude combinations. These are valid records and should not be treated as duplicates.

**Cleaning decision**: Remove only exact duplicate rows from `geolocation`. Records that share the same zip code but have different coordinates will be preserved.

In [55]:
# Remove exact duplicate rows from geolocation

geolocation = geolocation.drop_duplicates().reset_index(drop=True)

In [56]:
# Verify that exact duplicates were removed

print("Remaining duplicate rows:", geolocation.duplicated().sum())
print("Geolocation rows after cleaning:", len(geolocation))

Remaining duplicate rows: 0
Geolocation rows after cleaning: 738332


In [57]:
customers_clean = customers.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()
payments_clean = payments.copy()
reviews_clean = reviews.copy()
products_clean = products.copy()
sellers_clean = sellers.copy()
geolocation_clean = geolocation.copy()
category_translation_clean = category_translation.copy()
mql_clean = mql.copy()
closed_deals_clean = closed_deals.copy()

## 7. Correct data types

CSV files do not always preserve the appropriate data types. Date fields were loaded as text, while some Boolean fields were loaded as object. These fields are converted to appropriate Python data types before further analysis. Missing values are preserved during conversion.

In [58]:
# Convert order dates to datetime

order_date_columns = ["order_purchase_timestamp","order_approved_at","order_delivered_carrier_date","order_delivered_customer_date","order_estimated_delivery_date"]

for col in order_date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

In [59]:
# Convert order item shipping date to datetime

order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"],errors="coerce")

In [60]:
# Convert review dates to datetime

reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"],errors="coerce")

reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"],errors="coerce")

In [61]:
# Convert Marketing Funnel dates to datetime

mql["first_contact_date"] = pd.to_datetime(mql["first_contact_date"],errors="coerce")

closed_deals["won_date"] = pd.to_datetime(closed_deals["won_date"],errors="coerce")

In [62]:
# Verify date types

print(orders[order_date_columns].dtypes)
print(order_items["shipping_limit_date"].dtype)
print(reviews[["review_creation_date", "review_answer_timestamp"]].dtypes)
print(mql["first_contact_date"].dtype)
print(closed_deals["won_date"].dtype)

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object
datetime64[us]
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object
datetime64[us]
datetime64[us]


In [63]:
# Convert nullable Boolean fields

closed_deals["has_company"] = closed_deals["has_company"].astype("boolean")
closed_deals["has_gtin"] = closed_deals["has_gtin"].astype("boolean")

In [64]:
# Verify Boolean data types

closed_deals[["has_company", "has_gtin"]].dtypes

has_company    boolean
has_gtin       boolean
dtype: object

## 8. Standardise text and categorical fields

Text and categorical fields are checked for formatting inconsistencies that could create duplicate categories during analysis. Leading/trailing spaces, inconsistent capitalisation, and blank strings are inspected before deciding which transformations are necessary.

IDs are not modified.

In [65]:
# Check leading or trailing spaces in categorical fields

text_columns = {"customers": ["customer_city", "customer_state"],"orders": ["order_status"],"payments": ["payment_type"],"products": ["product_category_name"],
    "sellers": ["seller_city", "seller_state"],"geolocation": ["geolocation_city", "geolocation_state"],"mql": ["origin"],"closed_deals": ["business_segment",
        "lead_type","lead_behaviour_profile","average_stock","business_type"]}

dataframes = {"customers": customers,"orders": orders,"payments": payments,"products": products,"sellers": sellers,"geolocation": geolocation,"mql": mql,"closed_deals": closed_deals}

for name, columns in text_columns.items():
    df = dataframes[name]

    for col in columns:
        values = df[col].dropna().astype(str)
        spaces = (values != values.str.strip()).sum()

        if spaces > 0:
            print(f"{name}.{col}: {spaces} values with leading/trailing spaces")

geolocation.geolocation_city: 1 values with leading/trailing spaces


Only one value with leading/trailing spaces was found, in geolocation_city. This will be corrected during cleaning.

In [66]:
# Check inconsistent capitalisation in categorical fields

capitalisation_issues = 0

for name, columns in text_columns.items():
    df = dataframes[name]

    for col in columns:
        values = df[col].dropna().astype(str).str.strip()

        original_unique = values.nunique()
        lowercase_unique = values.str.lower().nunique()

        if original_unique != lowercase_unique:
            print(f"{name}.{col}: {original_unique} categories → {lowercase_unique} after lowercase")
            capitalisation_issues += 1

if capitalisation_issues == 0:
    print("No inconsistent capitalisation found.")

No inconsistent capitalisation found.


In [83]:
# Check blank strings in categorical fields

blank_string_issues = 0

for name, columns in text_columns.items():
    df = dataframes[name]

    for col in columns:
        blank_count = (df[col].dropna().astype(str).str.strip().eq("").sum())

        if blank_count > 0:
            print(f"{name}.{col}: {blank_count} blank strings")
            blank_string_issues += 1

if blank_string_issues == 0:
    print("No blank strings found.")

No blank strings found.


#### Findings and cleaning decision

No relevant formatting issues were found in the categorical fields. Only one value in geolocation_city contains leading/trailing spaces.

**Cleaning decision**: Remove leading/trailing spaces from geolocation_city. No other text standardisation is required.

In [68]:
# Remove leading/trailing spaces from geolocation city

geolocation["geolocation_city"] = geolocation["geolocation_city"].str.strip()

In [69]:
# Verify cleaning

(geolocation["geolocation_city"].dropna()!= geolocation["geolocation_city"].dropna().str.strip()).sum()

np.int64(0)

## 9. Table-specific cleaning decisions

Based on the missing-value investigation in Section 5, the required table-specific cleaning decisions are applied below. Valid records are preserved whenever missing values represent unavailable information rather than invalid observations.

In [70]:
# Verify missing values after table-specific cleaning decisions

print("Reviews columns:")
print(reviews.columns.tolist())

print("\nProducts missing categories:", products["product_category_name"].isna().sum())

print("\nMQL missing origin:", mql["origin"].isna().sum())

print("\nClosed deals missing categorical values:")
print(closed_deals[["business_segment", "lead_type", "lead_behaviour_profile", "business_type"]].isna().sum())

Reviews columns:
['review_id', 'order_id', 'review_score', 'review_creation_date', 'review_answer_timestamp']

Products missing categories: 0

MQL missing origin: 0

Closed deals missing categorical values:
business_segment          0
lead_type                 0
lead_behaviour_profile    0
business_type             0
dtype: int64


All planned categorical cleaning decisions have been applied. Missing values intentionally preserved in other fields remain unchanged.

## 10. Prepare ZIP-level geolocation reference

The geolocation table contains multiple coordinate observations for the same ZIP-code prefix.

A ZIP-level reference table is prepared to support later geographic analysis without creating duplicate rows when joining with customers or sellers.

Before defining the final reference, city and state consistency within each ZIP code is checked.

In [71]:
# Check how many cities and states are associated with each ZIP code

zip_location_consistency = (geolocation.groupby("geolocation_zip_code_prefix").agg(cities=("geolocation_city", "nunique"),states=("geolocation_state", "nunique")))

print("ZIP codes with multiple cities:",(zip_location_consistency["cities"] > 1).sum())

print("ZIP codes with multiple states:",(zip_location_consistency["states"] > 1).sum())

ZIP codes with multiple cities: 8555
ZIP codes with multiple states: 8


In [72]:
# Create ZIP-level geolocation reference

geolocation_zip = (geolocation.groupby("geolocation_zip_code_prefix", as_index=False).agg(geolocation_lat=("geolocation_lat", "median"),geolocation_lng=("geolocation_lng", "median")))

geolocation_zip.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng
0,1001,-23.549951,-46.634027
1,1002,-23.548228,-46.635247
2,1003,-23.548977,-46.635313
3,1004,-23.549550,-46.634771
4,1005,-23.549763,-46.636100


#### Findings and decision

Many ZIP-code prefixes are associated with multiple cities (8,555), while only 8 are associated with multiple states.

To avoid introducing arbitrary city or state labels, the ZIP-level reference will contain only the median latitude and longitude for each ZIP-code prefix.

The original city and state information remains available in the cleaned geolocation table.

## 11. Validate Marketing Funnel relationships

The relationship between `mql` and `closed_deals` is validated through `mql_id` before exporting the cleaned datasets.

The Marketing Funnel will be constructed later in SQL, where table relationships and conversion metrics will be analyzed.

In [73]:
# Check the relationship between MQLs and closed deals

print("MQL rows:", len(mql))
print("Unique MQL IDs:", mql["mql_id"].nunique())

print("\nClosed deal rows:", len(closed_deals))
print("Unique MQL IDs in closed deals:", closed_deals["mql_id"].nunique())

print("\nClosed deal MQLs found in MQL table:", closed_deals["mql_id"].isin(mql["mql_id"]).sum())

MQL rows: 8000
Unique MQL IDs: 8000

Closed deal rows: 842
Unique MQL IDs in closed deals: 842

Closed deal MQLs found in MQL table: 842


### Findings

All 842 closed deals correspond to unique MQLs present in the `mql` table.

This confirms that `mql_id` provides a valid relationship between Marketing Qualified Leads and converted leads. The Marketing Funnel will be constructed later in SQL.

## 12. Validate the relationship between Marketing and E-Commerce

The `seller_id` onnects converted Marketing leads in closed_deals with sellers in the E-Commerce dataset.

Before exporting the cleaned datasets, this relationship is validated to ensure that acquired sellers can later be connected to marketplace activity.

The analytical joins and seller performance metrics will be developed later in SQL.

In [74]:
# Check the relationship between closed deals and E-Commerce sellers

print("Closed deal sellers:", closed_deals["seller_id"].nunique())

print("Closed deal sellers found in sellers table:", closed_deals["seller_id"].isin(sellers["seller_id"]).sum())

print("Closed deal sellers found in order items:", closed_deals["seller_id"].isin(order_items["seller_id"]).sum())

Closed deal sellers: 842
Closed deal sellers found in sellers table: 380
Closed deal sellers found in order items: 380


In [75]:
# Check seller coverage in E-Commerce tables

print("E-Commerce sellers:", sellers["seller_id"].nunique())

print("E-Commerce sellers found in order items:", sellers["seller_id"].isin(order_items["seller_id"]).sum())

print("Order items linked to a valid seller:", order_items["seller_id"].isin(sellers["seller_id"]).sum())

E-Commerce sellers: 3095
E-Commerce sellers found in order items: 3095
Order items linked to a valid seller: 112650


In [76]:
# Measure overlap between acquired sellers and E-Commerce sellers

marketing_sellers = set(closed_deals["seller_id"])
ecommerce_sellers = set(sellers["seller_id"])

seller_overlap = marketing_sellers & ecommerce_sellers

print("Marketing sellers:", len(marketing_sellers))
print("E-Commerce sellers:", len(ecommerce_sellers))
print("Sellers present in both datasets:", len(seller_overlap))
print("Marketing sellers not found in E-Commerce:", len(marketing_sellers - ecommerce_sellers))

Marketing sellers: 842
E-Commerce sellers: 3095
Sellers present in both datasets: 380
Marketing sellers not found in E-Commerce: 462


#### Findings and decision

The documented relationships between the datasets were successfully validated. All E-Commerce sellers are represented in order_items, and seller_id provides the expected connection between the Marketing Funnel and E-Commerce data.

Of the 842 sellers acquired through Marketing, 380 are present in the available E-Commerce data, while 462 are not observed.

The reason for this partial coverage is not assumed at this stage and will be investigated during EDA.

## 13. Basic validity checks

Before exporting, key numeric and date fields are checked for impossible or suspicious values that could affect later business metrics.

Suspicious records are identified and documented here. Their potential impact on the analysis will be investigated during EDA.

In [84]:
# Describe key numeric columns before validity checks
order_items[["price", "freight_value"]].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [85]:
payments["payment_value"].describe()

count    103886.000000
mean        154.100380
std         217.494064
min           0.000000
25%          56.790000
50%         100.000000
75%         171.837500
max       13664.080000
Name: payment_value, dtype: float64

In [86]:
reviews["review_score"].describe()

count    99224.000000
mean         4.086421
std          1.347579
min          1.000000
25%          4.000000
50%          5.000000
75%          5.000000
max          5.000000
Name: review_score, dtype: float64

#### Findings

The descriptive statistics show that `price`, `freight_value` and `payment_value` are all right-skewed: the mean is higher than the median in each case, and the maximum value is far above the 75th percentile.

- `price`: median is 74.99, but the max is 6,735.00 — much higher than the 75th percentile (134.90).
- `freight_value`: median is 16.26, but the max is 409.68 — also far above the 75th percentile (21.15).
- `payment_value`: median is 100.00, but the max is 13,664.08 — the largest gap of the three, and the min of 0.00 matches the 9 records with `payment_value <= 0` already flagged above.

This pattern is a first signal that outliers exist in these columns. They are not treated here — outlier detection (IQR/z-score) will be done in Notebook 2, once these values are tied to specific sellers and business metrics like GMV.

`review_score` behaves differently: it's left-skewed, with a median of 5 and 75% of reviews at 4 or 5. Most customers rate their orders well, and only a smaller group of low scores pulls the mean down to 4.09. This is expected for review data and will matter later when comparing review scores between groups.

In [77]:
# Check invalid monetary values

print("Order items with price <= 0:", (order_items["price"] <= 0).sum())

print("Order items with freight value < 0:", (order_items["freight_value"] < 0).sum())

print("Payments with payment value <= 0:", (payments["payment_value"] <= 0).sum())

Order items with price <= 0: 0
Order items with freight value < 0: 0
Payments with payment value <= 0: 9


In [78]:
# Check values outside expected ranges

print("Invalid review scores:", (~reviews["review_score"].between(1, 5)).sum())

print("Order items with order_item_id <= 0:", (order_items["order_item_id"] <= 0).sum())

print("Payments with installments < 0:", (payments["payment_installments"] < 0).sum())

Invalid review scores: 0
Order items with order_item_id <= 0: 0
Payments with installments < 0: 0


In [79]:
# Check suspicious order date sequences

print("Approved before purchase:", (orders["order_approved_at"] < orders["order_purchase_timestamp"]).sum())

print("Delivered to carrier before purchase:", (orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"]).sum())

print("Delivered to customer before carrier:", (orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]).sum())

Approved before purchase: 0
Delivered to carrier before purchase: 166
Delivered to customer before carrier: 23


In [80]:
print("Delivered to customer before purchase:", (orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]).sum())

Delivered to customer before purchase: 0


### Findings

Most key numeric fields passed the basic validity checks.

However:
- 9 payment records have `payment_value <= 0`.
- 166 orders have a carrier date earlier than the purchase date.
- 23 orders have a customer delivery date earlier than the carrier date.

These records are flagged as suspicious but are not removed at this stage. They will be investigated during EDA before deciding whether they should be excluded from specific analyses.

## 14. Export processed data

The cleaned datasets are exported to `data/processed/` for use in the next stages of the project.

The original files in `data/raw/` remain unchanged.

The ZIP-level geolocation reference is also exported as a supporting dataset for later geographic analysis.

In [81]:
# Export cleaned datasets

customers.to_csv(f"{PROCESSED_PATH}/customers_clean.csv", index=False)
orders.to_csv(f"{PROCESSED_PATH}/orders_clean.csv", index=False)
order_items.to_csv(f"{PROCESSED_PATH}/order_items_clean.csv", index=False)
payments.to_csv(f"{PROCESSED_PATH}/payments_clean.csv", index=False)
reviews.to_csv(f"{PROCESSED_PATH}/reviews_clean.csv", index=False)
products.to_csv(f"{PROCESSED_PATH}/products_clean.csv", index=False)
sellers.to_csv(f"{PROCESSED_PATH }/sellers_clean.csv", index=False)
geolocation.to_csv(f"{PROCESSED_PATH}/geolocation_clean.csv", index=False)
category_translation.to_csv(f"{PROCESSED_PATH}/category_translation_clean.csv", index=False)
mql.to_csv(f"{PROCESSED_PATH}/mql_clean.csv", index=False)
closed_deals.to_csv(f"{PROCESSED_PATH}/closed_deals_clean.csv", index=False)

In [82]:
# Export ZIP-level geolocation reference
geolocation_zip.to_csv(f"{PROCESSED_PATH}/geolocation_zip_reference.csv",index=False)

print("Processed datasets exported successfully.")

Processed datasets exported successfully.


## 15. Final cleaning summary

The goal of this notebook was to understand the datasets, identify data-quality issues, and prepare reliable data for the analysis.

### Main cleaning decisions

- Date columns were converted to `datetime`.
- `has_company` and `has_gtin` were converted to nullable Boolean values.
- The misspelled product columns `lenght` were corrected to `length`.
- Review text columns were removed because they are not needed for the planned analysis.
- English product category names were added.
- Missing product categories and MQL origins were labelled as `"unknown"`.
- Missing categorical seller information was labelled as `"missing"` instead of guessing values.
- Exact duplicate rows were removed from `geolocation`.
- The only formatting issue found, a leading/trailing space in `geolocation_city`, was corrected.
- A ZIP-level geolocation reference was created using median latitude and longitude for each ZIP-code prefix.

### What was preserved

Not every missing or repeated value represents bad data, so valid information was kept whenever possible.

- Missing values were preserved when they represent unavailable information rather than invalid records.
- Repeated IDs caused by valid one-to-many relationships were kept.
- Missing order dates were not imputed when they were consistent with the order status.
- Suspicious payment values and date sequences were identified but not automatically removed. Their impact will be investigated during EDA.
- The original city and state information in `geolocation` was preserved alongside the ZIP-level reference.
- The original files in `data/raw/` remain unchanged.

### Relationships validated

The main relationships needed for later analysis were also checked.

- The E-Commerce tables are consistently connected through their documented IDs.
- All 842 closed deals correspond to unique MQLs present in the `mql` table.
- Marketing and E-Commerce can be connected through `seller_id`.
- Of the 842 Marketing-acquired sellers, 380 are observed in the available E-Commerce data, while 462 are not. The reason for this partial coverage is not assumed at this stage and will be investigated during EDA.

### Output

The cleaned datasets were exported to `data/processed/`, together with the ZIP-level geolocation reference.

The data is now ready for **Notebook 02 — Exploratory Data Analysis & Business Analysis**, where the focus moves from preparing the data to understanding the business patterns behind seller acquisition, marketplace performance, delivery reliability, and customer satisfaction. 

The next step is to explore marketplace performance, customer experience, delivery, sellers and Marketing Funnel performance, while investigating the open questions identified during cleaning.